In [1]:
#!pip install langchain langchain-core langchain-community pypdf pymupdf

In [2]:
#!pip install sentence-transformers chromadb

In [3]:
from langchain_core.documents import Document

### A sample document

In [4]:
sample_doc = Document(
    page_content = "hello world",
    metadata = {"source" : "google.com"}
)

In [5]:
print(sample_doc)

page_content='hello world' metadata={'source': 'google.com'}


In [6]:
type(sample_doc)

langchain_core.documents.base.Document

### working with text data

In [7]:
from langchain_community.document_loaders.text import TextLoader

loader = TextLoader("data/python.txt", encoding="utf-8")

C:\Users\prash\AppData\Local\Temp\ipykernel_13684\3383052197.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader


In [8]:
document = loader.load()

In [9]:
document

[Document(metadata={'source': 'data/python.txt'}, page_content='Python is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for experienced developers. Its clean and concise syntax allows programmers to write fewer lines of code compared to many other languages, enhancing productivity and maintainability. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, which makes it versatile for a wide range of applications.\nSome key features and benefits of Python include:\n* Ease of Learning: Simple syntax and readability make Python beginner-friendly.\n* Versatility: Suitable for web development, data analysis, artificial intelligence, machine learning, scientific computing, automation, and mo

### working with pdfs

In [10]:
from langchain_community.document_loaders.pdf import PyPDFLoader

pdf_loader = PyPDFLoader("data/pdfs/research1.pdf")

document = pdf_loader.load()

In [11]:
#document

In [12]:
from langchain_community.document_loaders.pdf import PyMuPDFLoader

pdf_loader = PyMuPDFLoader("data/pdfs/research1.pdf")

document = pdf_loader.load()

In [13]:
#document

## Ingestion Pipeline

In [14]:
# Data => Documents

import os
from langchain_community.document_loaders.pdf import PyPDFLoader

In [15]:
def load_all_pdfs():
    folder_path= "data/pdfs"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()

            all_docs.extend(doc)
            num_docs +=1

    print("total pdfs = ", num_docs)
    print("total pages = ", len(all_docs))

    return all_docs

In [16]:
all_pdf_documents = load_all_pdfs()

total pdfs =  2
total pages =  32


In [17]:
type(all_pdf_documents[1])

langchain_core.documents.base.Document

In [18]:
# CHUNKS

#!pip install langchain_text_splitters

In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter as RCT

In [20]:
def split_docs(documents, chunk_size=500, chunk_overlap=50):

    text_splitter = RCT(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(documents)

    return chunked_docs

    

In [21]:
chunks = split_docs(all_pdf_documents)

In [22]:
len(chunks)

320

In [23]:
#chunks

### EMBEDDING (using Sentence-Transformers)

In [24]:
from sentence_transformers import SentenceTransformer

In [25]:
class EmbeddingManager:
    def __init__(self,model_name = "all-MiniLM-L6-v2"): # model to create embeddings

        #loading model
        self.model_name = model_name
        print("loading model.....", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions=", self.model.get_embedding_dimension())


    def generate_embedding(self,text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embedding shape: ", embeddings.shape)
        return embeddings
        

In [26]:
embedding_manager = EmbeddingManager()

loading model..... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedding dimensions= 384


### Vector Store

In [27]:
import chromadb
import uuid

In [28]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name = "pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None # an entity which connects to the data store

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok= True)

        #creating a client

        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # creating the collection

        self.collection = self.client.get_or_create_collection(
            name = self.collection_name,
            metadata = {"description" : "vector store collection for pdf embeddings in RAG"}
        )

        print("initialised the vector store with collection : ", self.collection_name)
        print("docs in collection:", self.collection.count())


    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")

        #store =>> ids,  embeddings , documents, metadata

        ids =[]
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents , embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids = ids,
                metadatas = all_metadata,
                documents = documents_content,
                embeddings = embeddings_list
            )

        print("total documents added in the vector store=", len(documents_content))
        print("docs in collection:",self.collection.count())

In [29]:
vector_store = VectorStoreManager()

initialised the vector store with collection :  pdf_documents
docs in collection: 3200


In [30]:
# data => document => chunks => embeddings => store in vector database

In [31]:
texts = [doc.page_content for doc in chunks]

embedding = embedding_manager.generate_embedding(texts)

vector_store.add_documents(chunks, embedding)

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

embedding shape:  (320, 384)
total documents added in the vector store= 320
docs in collection: 3520


### Retrieval Pipeline

In [32]:
from sklearn.metrics.pairwise import cosine_similarity

In [33]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self, query, top_k = 5, score_threshold = 0.0):
        # query => embedding
        query_embedding = self.embedding_manager.generate_embedding([query])[0]

        results = self.vector_store.collection.query(
            query_embeddings = [query_embedding.tolist()],
            n_results =top_k
        )

        # cosine similirity

        retrieved_docs=[]

        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance # formula

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document":document,
                        "metadata":metadata,
                        "distance":distance,
                        "similarity_score": similarity_score,
                        "rank":i+1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("no documents found")

        return retrieved_docs

                

In [34]:
rag_retriever = RAGRetriever(embedding_manager,vector_store )

In [35]:
rag_retriever.retrieve("what is RAG")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding shape:  (1, 384)
retrieved 5 documents


[{'id': 'doc_c9a2e2e7-6019-4f5f-b1b5-5d83730681b9',
  'document': 'and speculate on upcoming trends and innovations.\nOur contributions are as follows:\n• In this survey, we present a thorough and systematic\nreview of the state-of-the-art RAG methods, delineating\nits evolution through paradigms including naive RAG,\narXiv:2312.10997v5  [cs.CL]  27 Mar 2024',
  'metadata': {'author': '',
   'source': 'data/pdfs\\research2.pdf',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'title': '',
   'page_label': '1',
   'page': 0,
   'subject': '',
   'keywords': '',
   'doc_index': 87,
   'moddate': '2024-03-28T00:54:45+00:00',
   'content_length': 288,
   'trapped': '/False',
   'creator': 'LaTeX with hyperref',
   'total_pages': 21,
   'creationdate': '2024-03-28T00:54:45+00:00',
   'producer': 'pdfTeX-1.40.25'},
  'distance': 0.4226759076118469,
  'similarity_score': 0.5773240923881531,
  'rank': 1},
 {'id': 'doc_cd3e44af-

## Integrate with LLMs

### OpenAI - GPT

In [36]:
import os
API_KEY_OPENAI = os.environ["OPENAI_API_KEY"]

In [37]:
#!pip install langchain-openai

In [38]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    openai_api_key = API_KEY_OPENAI,
    model = "gpt-5.5",
    temperature = 0.1,
    max_tokens=1024

)

In [39]:
# generate our Retrieval Augmented output

def generate_output(query, rag_retriever, llm, top_k = 3):
    results = rag_retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query

    prompt = f"""use given context to generate the answer for the query
            Context: {context}
            Query: {query}"""

    
    response = llm.invoke(prompt)
    return response.content

In [40]:
#answer = generate_output("what is RAG?", rag_retriever, llm)

In [41]:
print("good_boy")

good_boy


In [42]:
#!pip install langchain-groq

## Groq

In [43]:
import os
API_Key_GROQ = os.environ["GROQ_API_KEY"]

In [44]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key = API_Key_GROQ,
    model = "openai/gpt-oss-120b",
    temperature = 0.1,
    max_tokens=1024
)

In [45]:
# generate our Retrieval Augmented output

def generate_output(query, rag_retriever, llm, top_k = 3):
    results = rag_retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query

    prompt = f"""use given context to generate the answer for the query
            Context: {context}
            Query: {query}"""

    
    response = llm.invoke([prompt.format(context=context, query=query)]) # groqexpects a list as prompt
    return response.content

In [46]:
answer = generate_output("what is RAG?", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding shape:  (1, 384)
retrieved 3 documents


In [47]:
print(answer)

**RAG (Retrieval‑Augmented Generation)** is a hybrid AI architecture that combines two complementary components:

| Component | What it does | Why it’s useful |
|-----------|--------------|-----------------|
| **Retriever** | Searches an external knowledge source (e.g., a document corpus, database, or the web) to fetch the most relevant passages for a given query. | Provides up‑to‑date, factual, and domain‑specific information that the language model may not have seen during pre‑training. |
| **Generator** | A large language model (LLM) that takes the retrieved passages (often concatenated with the original prompt) and produces a natural‑language response. | Leverages the LLM’s fluency and reasoning abilities while grounding its output in concrete evidence. |

### How RAG works (high‑level flow)

1. **Input query** → sent to the retriever.  
2. **Retriever** returns *k* top‑ranked documents/passages (e.g., via BM25, dense embeddings, or hybrid methods).  
3. The retrieved texts are **c

In [48]:
#!pip install langchain-anthropic